Method: https://proghead231.github.io/agricultural-land-abandonment-nepal/
1. Data acquisition and preprocessing
2. Create agricultural land (AL) objects
3. Calculate annual AL probability
4. Apply landTrendr to the time series of AL probabilities
5. Identify structural breaks and classify eac AL objects to abandoned (AAL), fallow (FAL) or recultivate (RAL)
6. Validate results

## Step 1: Create AL objects
Given the uncertainity in the AL classification of the land cover datasets by FRTC I will do segmentation using some algo
1. Get landsat-SR images from 2000, 2010 and 2020, preprocess them and stack them 
2. Get texture metrics from the one image, combine the metrics into one texture image and add to the stack (Using GLCM metrics here)
3. Perform segmentation and check

In [1]:
import ee
# ee.Authenticate()
ee.Initialize(project="ee-joshisur231")
import geemap
from helpers import config
from helpers import utils
Map = geemap.Map()
import os
from helpers import landsat_composites

In [2]:
roi = ee.FeatureCollection("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/ob_segmentation_test_area")
config.ROI = roi

dem = ee.Image("USGS/SRTMGL1_003").rename("z")
terrain = ee.Terrain.products(dem)
seg_bands = ee.List(["blue", "green", "red", "nir", "swir1", "swir2", "ndvi", "ndmi", "ndvi_shade", "ndvi_savg"])
indices_formula = {
    'NDVI': 'clamp((b("nir") - b("red")) / (b("nir") + b("red")), -1, 1)',
    'EVI': (
        'clamp(2.5 * ((b("nir") - b("red")) / '
        '     clamp(b("nir") + 6 * b("red") - 7.5 * b("blue") + 1, 0, 10)), '
        '   -1.0, 1.0)'
    ),
    'NBR': 'clamp((b("nir") - b("swir2")) / (b("nir") + b("swir2")), -1, 1)',
    'NDWI': 'clamp((b("green") - b("nir")) / (b("green") + b("nir")), -1, 1)',
    'NDMI': 'clamp((b("nir") - b("swir1")) / (b("nir") + b("swir1")), -1, 1)',
    'BAI': '1.0 / (pow(0.1 - b("red"), 2) + pow(0.06 - b("nir"), 2))',
    'MSAVI': '(2 * b("nir") + 1 - sqrt(pow(2 * b("nir") + 1, 2) - 8 * (b("nir") - b("red")))) / 2.0',
}
def add_indices(image):
    ndvi = image.expression(indices_formula["NDVI"]).rename("ndvi")
    ndmi = image.expression(indices_formula["NDMI"]).rename("ndmi")
    evi = image.expression(indices_formula["EVI"]).rename("evi")
    msavi = image.expression(indices_formula["MSAVI"]).rename("msavi")
    return image.addBands(ndvi).addBands(ndmi).addBands(evi).addBands(msavi)

l_winter2002_image = utils.get_processed_landsat_collection("LANDSAT/LE07/C02/T1_L2", config.ROI, ["2002-12-01", "2003-04-01"], utils.mask_clouds_landsat75, utils.apply_scale_factors).median().select(config.L75_ORIGINAL_BAND_NAMES).rename(config.L75_NEW_BAND_NAMES)
l_winter2002_image = add_indices(l_winter2002_image)
high_val_winter2002 = ee.Number(utils.calc_image_stats(l_winter2002_image, config.ROI, config.SCALE).get("high"))
l_winter2002_image = utils.add_scaled_glcm(l_winter2002_image, config.ROI, config.SCALE, high_val_winter2002).select(seg_bands).rename(seg_bands.map(lambda band_name: ee.String(band_name).cat("_2002")))

terrain = utils.prepare_terrain_seg(terrain, config.ROI, scale = config.SCALE, high_val=high_val_winter2002)

l_winter2010_image = utils.get_processed_landsat_collection("LANDSAT/LT05/C02/T1_L2", config.ROI, ["2010-12-01", "2011-03-01"],utils.mask_clouds_landsat75, utils.apply_scale_factors).median().select(config.L75_ORIGINAL_BAND_NAMES).rename(config.L75_NEW_BAND_NAMES)
l_winter2010_image = add_indices(l_winter2010_image)
high_val_winter2010 = ee.Number(utils.calc_image_stats(l_winter2010_image, config.ROI, config.SCALE).get("high"))
l_winter2010_image = utils.add_scaled_glcm(l_winter2010_image, config.ROI, config.SCALE, high_val_winter2010).select(seg_bands).rename(seg_bands.map(lambda band_name: ee.String(band_name).cat("_2010")))

l_preMons_image = utils.get_processed_landsat_collection("LANDSAT/LC08/C02/T1_L2", config.ROI, ["2017-03-01", "2017-06-01"], utils.mask_clouds_landsat8, utils.apply_scale_factors).median().select(config.L8_ORIGINAL_BAND_NAMES).rename(config.L8_NEW_BAND_NAMES)
l_preMons_image = add_indices(l_preMons_image)
high_val_preMons = ee.Number(utils.calc_image_stats(l_preMons_image, config.ROI, config.SCALE).get("high"))
l_preMons_image = utils.add_scaled_glcm(l_preMons_image, config.ROI, config.SCALE, high_val_preMons).select(seg_bands).rename(seg_bands.map(lambda band_name: ee.String(band_name).cat("_2017")))

seg_image_stack = l_winter2002_image.addBands(l_winter2010_image).addBands(l_preMons_image).addBands(terrain)

In [5]:
seg_image_stack.bandNames().getInfo()

['blue_2002',
 'green_2002',
 'red_2002',
 'nir_2002',
 'swir1_2002',
 'swir2_2002',
 'ndvi_2002',
 'ndmi_2002',
 'ndvi_shade_2002',
 'ndvi_savg_2002',
 'blue_2010',
 'green_2010',
 'red_2010',
 'nir_2010',
 'swir1_2010',
 'swir2_2010',
 'ndvi_2010',
 'ndmi_2010',
 'ndvi_shade_2010',
 'ndvi_savg_2010',
 'blue_2017',
 'green_2017',
 'red_2017',
 'nir_2017',
 'swir1_2017',
 'swir2_2017',
 'ndvi_2017',
 'ndmi_2017',
 'ndvi_shade_2017',
 'ndvi_savg_2017',
 'slope',
 'northness',
 'eastness']

In [52]:
crop_prob_dict_list = ee.data.listAssets("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/crop_prob")["assets"]
crop_list = ee.List([ee.Image(dict["id"]).gt(0.5).unmask(0) for dict in crop_prob_dict_list])
num_years = crop_list.size()
indices = ee.List.sequence(0, num_years.subtract(3))

def cons_3yr_crop(i):
  i = ee.Number(i)
  y1 = ee.Image(crop_list.get(i))
  y2 = ee.Image(crop_list.get(i.add(1)))
  y3 = ee.Image(crop_list.get(i.add(2)))
  return y1.And(y2).And(y3)

cons_3yr_col = ee.ImageCollection(indices.map(cons_3yr_crop))
active_crop = cons_3yr_col.max().rename("active_crop")

In [53]:
target_crs = "EPSG:32645"
target_scale = 30

seeds_test = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
connectivity_test = [4, 8]
neighbor_test = [6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26]
compactness_test = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

paired_seedsNneighborhood = list(zip(seeds_test, neighbor_test))

combination_count=0
for seed, neighborhood in paired_seedsNneighborhood:
    for connectivity in connectivity_test:
        for compactness in compactness_test:
            combination_count += 1
            print(f"Combo #{combination_count:02d} -> Seeds: {seed} | Neighbors: {neighborhood} | Connectivity: {connectivity} | Compactness: {compactness}")
            seeds = ee.Algorithms.Image.Segmentation.seedGrid(seed, "hex").reproject(crs=target_crs, scale=target_scale)
            snic = ee.Algorithms.Image.Segmentation.SNIC(
                image= seg_image_stack.reproject(crs=target_crs, scale=target_scale),
                connectivity= connectivity,
                neighborhoodSize= neighborhood,
                seeds= seeds,
                compactness=compactness
            )
            def extract_and_tag(feature):
                region_loc = feature.get('geoReg')
                combined_reducer = ee.Reducer.mean().combine(reducer2=ee.Reducer.variance(), sharedInputs=True)
                local_contours = snic.select('clusters').reduceToVectors(
                    geometry=feature.geometry(), 
                    scale=target_scale, 
                    maxPixels=1e13,
                    crs=target_crs
                )
                
                #Selecting only ag segment
                
                local_stats = seg_image_stack.addBands(active_crop).reduceRegions(
                    collection=local_contours,
                    scale=target_scale,
                    crs=target_crs,
                    reducer=combined_reducer,
                    tileScale=4
                )
                
                
                return local_stats.map(lambda cluster: cluster.set('geoReg', region_loc))


            cluster_stats = ee.FeatureCollection(config.ROI).map(extract_and_tag).flatten()

            filename = f"s-{seed}_n-{neighborhood}_cn-{connectivity}_cm-{str(compactness)[-1]}"

            geemap.ee_export_vector_to_drive(
                collection = cluster_stats,
                description = filename,
                fileFormat = "GeoJSON",
                folder="aal"
            )


Combo #01 -> Seeds: 3 | Neighbors: 6 | Connectivity: 4 | Compactness: 0.1
Exporting s-3_n-6_cn-4_cm-1... Please check the Task Manager from the JavaScript Code Editor.
Combo #02 -> Seeds: 3 | Neighbors: 6 | Connectivity: 4 | Compactness: 0.2
Exporting s-3_n-6_cn-4_cm-2... Please check the Task Manager from the JavaScript Code Editor.
Combo #03 -> Seeds: 3 | Neighbors: 6 | Connectivity: 4 | Compactness: 0.3
Exporting s-3_n-6_cn-4_cm-3... Please check the Task Manager from the JavaScript Code Editor.
Combo #04 -> Seeds: 3 | Neighbors: 6 | Connectivity: 4 | Compactness: 0.4
Exporting s-3_n-6_cn-4_cm-4... Please check the Task Manager from the JavaScript Code Editor.
Combo #05 -> Seeds: 3 | Neighbors: 6 | Connectivity: 4 | Compactness: 0.5
Exporting s-3_n-6_cn-4_cm-5... Please check the Task Manager from the JavaScript Code Editor.
Combo #06 -> Seeds: 3 | Neighbors: 6 | Connectivity: 4 | Compactness: 0.6
Exporting s-3_n-6_cn-4_cm-6... Please check the Task Manager from the JavaScript Code 

In [61]:
# vis_params = {"min":0, "max":0.3, "bands": ["red", "green", "blue"]}
# Map.addLayer(l_marMay_image.clip(config.ROI), {}, "l_marMay")
# # Map.addLayer(ee.ImageCollection([l_octNov_image.clip(config.ROI), l_octNov_image2.clip(config.ROI)]).mosaic(), vis_params, "l_octNov2")
# # Map.addLayer(ee.ImageCollection(l_octNov_image2.clip(config.ROI)), vis_params, "l_octNov2")
# Map.addLayer(l_octNov_image.clip(config.ROI), vis_params, "l_octNov")
# Map.addLayer(l_decFeb_image.clip(config.ROI), {}, "l_decFeb")
# Map

In [1]:
# for year in range(1990, 2005):
#     image = utils.get_processed_landsat_collection("LANDSAT/LT05/C02/T1_L2", config.ROI, [f"{year}-10-01", f"{year}-12-01"],utils.mask_clouds_landsat75, utils.apply_scale_factors).median().select(config.L75_ORIGINAL_BAND_NAMES).rename(config.L75_NEW_BAND_NAMES).clip(config.ROI)
#     count = image.select(0).reduceRegion(reducer=ee.Reducer.count(), scale=500, geometry=config.ROI.geometry())
#     print(f"{year}: {count.getInfo()}")

In [ ]:
target_crs = "EPSG:32645"
target_scale = 30
#s-11_n-22_cn-8_cm-1
seeds = ee.Algorithms.Image.Segmentation.seedGrid(11, "hex").reproject(crs=target_crs, scale=target_scale)
snic = ee.Algorithms.Image.Segmentation.SNIC(
  image= seg_image_stack.reproject(crs=target_crs, scale=target_scale),
  connectivity= 8,
  neighborhoodSize= 22,
  seeds= seeds,
  compactness=0.1
)
contours = snic.select('clusters') \
    .reduceToVectors(geometry=config.ROI, scale=30, maxPixels=1e13)

In [57]:
def get_s2_reference(roi, date_range):
    # 1. Define Sentinel-2 Cloud Mask
    def mask_s2_clouds(image):
        qa = image.select('QA60')
        # Bits 10 and 11 are clouds and cirrus, respectively.
        cloudBitMask = 1 << 10
        cirrusBitMask = 1 << 11
        mask = qa.bitwiseAnd(cloudBitMask).eq(0) \
            .And(qa.bitwiseAnd(cirrusBitMask).eq(0))
        return image.updateMask(mask).divide(10000) # Apply scaling (0.0001) here

    # 2. Fetch Collection
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterDate(date_range[0], date_range[1]) \
        .filterBounds(roi) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
        .map(mask_s2_clouds) \
        .median() \
        .clip(roi)

    return s2

s2_dates = ["2017-01-01", "2017-12-28"]

# Get the image
s2_2020_image = get_s2_reference(config.ROI, s2_dates)

vis_params_s2 = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2'],
}
Map.addLayer(s2_2020_image.clip(config.ROI), vis_params_s2, 'Sentinel-2 (10m) Ref', False)

vis_params = {"min":0, "max":0.3, "bands": ["nir", "swir1", "r"]} 

# Map.addLayer(ee.ImageCollection([l_winter2010_image.select(["green_2010", "red_2010", "nir_2010", "swir1_2010"]).rename(["g", "r", "nir", "swir1"]),l.select(["g_2000", "r_2000", "nir_2000", "swir1_2000"]).rename(["g", "r", "nir", "swir1"]),l8_2020_image.select(["g_2020", "r_2020", "nir_2020", "swir1_2020"]).rename(["g", "r", "nir", "swir1"])]).mean(), vis_params, "l8")
# Map.addLayer(l_winter2010_image.select(["green_2010", "red_2010", "nir_2010", "swir1_2010"]).rename(["g", "r", "nir", "swir1"]), vis_params, "winter_2000")
Map.addLayer(l_preMons_image.select(["green_2017", "red_2017", "nir_2017", "swir1_2017"]).rename(["g", "r", "nir", "swir1"]).clip(config.ROI), vis_params, "preMons2017")

# Map.addLayer(seg_image_stack.select("nir_savg_2020"), {"min":0, "max":0.3}, 'savg')
# Map.addLayer(seg_image_stack.select("nir_shade_2020"), {"min":0, "max":0.3}, 'shade')
Map.addLayer(
    contours.style(color='ffffff', width=1, fillColor='00000000'), 
    {}, 
    'Segment Boundaries'
)
# Map.addLayer(snic.select("clusters"), {}, "seg")
# Map.addLayer(l5_2010_image, vis_params, "l5")

Map

Map(bottom=3809.0, center=[24.966140159912975, 91.03271484375001], controls=(WidgetControl(options=['position'…

In [13]:
# cluster_stats = seg_image_stack.addBands(config.ROI.reduceToImage(properties=["geoCode"], reducer = ee.Reducer.first())).reduceRegions(collection=contours, scale=target_scale, crs=target_crs, reducer=ee.Reducer.mean())
# stats_df = geemap.ee_to_df(cluster_stats, remove_geom=False)

In [1]:
import geopandas as gpd

In [4]:
df=gpd.read_file(r"outputs\obia_results\parameter_tuning_inputs\s-3_n-6_cn-4_cm-1.geojson")

In [7]:
df.head().columns

Index(['id', 'active_crop_mean', 'active_crop_variance', 'blue_2002_mean',
       'blue_2002_variance', 'blue_2010_mean', 'blue_2010_variance',
       'blue_2017_mean', 'blue_2017_variance', 'count', 'eastness_mean',
       'eastness_variance', 'geoReg', 'green_2002_mean', 'green_2002_variance',
       'green_2010_mean', 'green_2010_variance', 'green_2017_mean',
       'green_2017_variance', 'label', 'ndmi_2002_mean', 'ndmi_2002_variance',
       'ndmi_2010_mean', 'ndmi_2010_variance', 'ndmi_2017_mean',
       'ndmi_2017_variance', 'ndvi_2002_mean', 'ndvi_2002_variance',
       'ndvi_2010_mean', 'ndvi_2010_variance', 'ndvi_2017_mean',
       'ndvi_2017_variance', 'ndvi_savg_2002_mean', 'ndvi_savg_2002_variance',
       'ndvi_savg_2010_mean', 'ndvi_savg_2010_variance', 'ndvi_savg_2017_mean',
       'ndvi_savg_2017_variance', 'ndvi_shade_2002_mean',
       'ndvi_shade_2002_variance', 'ndvi_shade_2010_mean',
       'ndvi_shade_2010_variance', 'ndvi_shade_2017_mean',
       'ndvi_shade_201

## Checking suitable segmentation parameters

In [7]:
import pandas as pd
import os

feat_impor_path = r"outputs/random_forest/feature_importance"
file_list = os.listdir(feat_impor_path)
top_df_list = [pd.read_csv(os.path.join(feat_impor_path, file)).sort_values("importance", ascending=False) for file in file_list]
long_df = pd.concat(top_df_list, ignore_index=True)
mean_df = long_df.groupby("feature").agg({"importance": "mean"})
print(mean_df.sort_values("importance", ascending=False).head(10))

              importance
feature                 
swir2_stdDev    0.102985
elev            0.093481
ndvi_stdDev     0.078720
ndmi_stdDev     0.069420
msavi_stdDev    0.052288
green_p75       0.046819
evi_stdDev      0.046651
green_mean      0.041947
green_median    0.039603
msavi_mean      0.026974


In [19]:
tex_col = ['ndvi_mean_savg', 'ndvi_mean_shade', 'ndvi_mean_contrast',
       'ndvi_mean_dvar', 'ndvi_mean_prom', 'ndvi_mean_var',
       'ndvi_mean_idm', 'ndvi_mean_dent', 'ndvi_mean_asm',
       'ndvi_mean_svar', 'ndvi_mean_imcorr2', 'ndvi_mean_imcorr1',
       'ndvi_mean_inertia', 'ndvi_mean_diss', 'ndvi_mean_ent',
       'ndvi_mean_corr', 'ndvi_mean_sent']
long_df_tex = long_df[long_df["feature"].isin(tex_col)]

In [24]:
print(long_df_tex.groupby("feature").agg({"importance": "mean"}).sort_values("importance", ascending=False))

                    importance
feature                       
ndvi_mean_savg        0.026749
ndvi_mean_shade       0.001045
ndvi_mean_asm         0.000879
ndvi_mean_inertia     0.000875
ndvi_mean_imcorr2     0.000869
ndvi_mean_var         0.000806
ndvi_mean_contrast    0.000767
ndvi_mean_prom        0.000760
ndvi_mean_svar        0.000740
ndvi_mean_idm         0.000716
ndvi_mean_corr        0.000686
ndvi_mean_dent        0.000678
ndvi_mean_ent         0.000676
ndvi_mean_dvar        0.000655
ndvi_mean_imcorr1     0.000567
ndvi_mean_diss        0.000554
ndvi_mean_sent        0.000552
